In [2]:
from tqdm import tqdm
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
import pandas as pd
from bs4 import BeautifulSoup as bs

driver = webdriver.Chrome()
driver.get('http://www.naver.com')

# 검색창 위치 찾아서 검색하기
search = driver.find_element(By.ID, 'query')
search.click()
search.send_keys('신혼' + "\n")
time.sleep(1)

# 지식인 탭 클릭하기
driver.find_element(
    By.CSS_SELECTOR,
    '#lnb > div.lnb_group > div > div.lnb_nav_area._nav_area_root > div > div.api_flicking_wrap._conveyer_root > div:nth-child(6) > a'
).click()
time.sleep(1)

# 옵션 클릭
option = driver.find_element(By.XPATH, '//*[@id="snb"]/div[1]/div/div[2]/a')
option.click()
time.sleep(1)

# 직접입력
direct_input = driver.find_element(By.XPATH, '//*[@id="snb"]/div[2]/ul/li[1]/div/div[1]/a[9]')
direct_input.click()
time.sleep(1)

# 2024 클릭
year = driver.find_element(By.XPATH, '//*[@id="snb"]/div[2]/ul/li[1]/div/div[2]/div[2]/div[1]/div/div/div/ul/li[35]/a')
year.click()
time.sleep(1)

# 적용 버튼 클릭
set_button = driver.find_element(By.XPATH, '//*[@id="snb"]/div[2]/ul/li[1]/div/div[2]/div[3]/button')
set_button.click()
time.sleep(1)

In [3]:
body = driver.find_element(By.CSS_SELECTOR, 'body')

max_scroll = 100
scroll_count = 0

for i in tqdm(range(max_scroll), desc="스크롤 진행 중", unit="회"):
    last = driver.page_source

    body.send_keys(Keys.END)
    time.sleep(1.5)

    new = driver.page_source
    scroll_count += 1

    if last == new:
        print(f"더 이상 로딩 없음. {scroll_count}회에서 종료")
        break

print(f"스크롤 완료: 총 {scroll_count}회")

스크롤 진행 중:  97%|█████████▋| 97/100 [03:32<00:06,  2.19s/회]

더 이상 로딩 없음. 98회에서 종료
스크롤 완료: 총 98회


In [4]:
soup = bs(driver.page_source, 'html.parser')

url_list = soup.select('a[href*="qna/detail"]')

url_list_result = [i['href'] for i in url_list if i.get('href')]

# 중복 제거
url_list_result = list(dict.fromkeys(url_list_result))

print(f"수집된 URL 수: {len(url_list_result)}")

수집된 URL 수: 1868


In [5]:
rows = []

for index, url in enumerate(tqdm(url_list_result, desc="상세 페이지 수집 중", unit="건")):
    driver.get(url)
    time.sleep(1.5)

    # 답변 더보기 클릭
    try:
        while True:
            body = driver.find_element(By.CSS_SELECTOR, 'body')
            body.send_keys(Keys.END)
            time.sleep(0.5)

            next_button = driver.find_element(By.ID, 'nextPageButton')
            next_button.click()
            time.sleep(1)

    except:
        pass

    soup = bs(driver.page_source, 'html.parser')

    # 제목
    title_tag = soup.select_one('div.endTitleSection')
    if title_tag:
        title = title_tag.get_text(strip=True)
        title = title.replace('\n', '').replace('\t', '')
        title = title.replace('질문', '', 1).strip()
    else:
        title = ''

    # 본문
    contents_tag = soup.select_one('div.questionDetail')
    contents = contents_tag.get_text(" ", strip=True) if contents_tag else ''

    # 답변
    answer_tags = soup.select('div.se-main-container')
    final_return = [i.get_text(" ", strip=True) for i in answer_tags]

    # 날짜
    date_tag = soup.select_one('span.c-userinfo__time')
    date = date_tag.get_text(strip=True).rstrip('.') if date_tag else ''

    rows.append({
        '제목': title,
        '본문': contents,
        '답변': final_return,
        '날짜': date,
        'URL': url
    })

df = pd.DataFrame(rows)

print(f"최종 수집 데이터 수: {len(df)}")
df.head()

상세 페이지 수집 중: 100%|██████████| 1868/1868 [1:50:32<00:00,  3.55s/건]

최종 수집 데이터 수: 1868


,제목,본문,답변,날짜,URL
0,SH 공공임대 신혼부부...,SH 공공임대 신혼부부 유형으로 청약 지원하려고 하는데 무주택자 기준으로 질문드립니...,[세계사이버대학 부동산금융자산학과장 강병기입니다 . ​ 걱정하지 않아도 됩니다. 예...,,https://kin.naver.com/qna/detail.naver?answerN...
1,SH 공공임대 신혼부부...,SH 공공임대 신혼부부 유형으로 청약 지원하려고 하는데 무주택자 기준으로 질문드립니...,[세계사이버대학 부동산금융자산학과장 강병기입니다 . ​ 걱정하지 않아도 됩니다. 예...,,https://kin.naver.com/qna/detail.naver?answerN...
2,버팀목 전세대출 신혼부부,안녕하세요 버팀목 전세대출을 받으려고 하는데 제가 6월말 퇴사 예정이고 신혼집은 1...,[그렇습니다 배우자님 외벌이소득조건 인정됩니다 퇴사후 이직하신다면 소득합산도 가능합...,,https://kin.naver.com/qna/detail.naver?answerN...
3,버팀목 전세대출 신혼부부,안녕하세요 버팀목 전세대출을 받으려고 하는데 제가 6월말 퇴사 예정이고 신혼집은 1...,[그렇습니다 배우자님 외벌이소득조건 인정됩니다 퇴사후 이직하신다면 소득합산도 가능합...,,https://kin.naver.com/qna/detail.naver?answerN...
4,신혼부부 첫주택구입 대출 질문,디딤돌 대출로 신청할 예정입니다. 처음으로 대출하려고 하니 모르는게 많네요. 여러분...,[대출 신청전에 혼인관계이시면 대출 차주만 무주택 세대주 이면 됩니다. ( 배우자가...,,https://kin.naver.com/qna/detail.naver?answerN...


In [6]:
df.to_csv('naver_kin_신혼_2024.csv', index=False, encoding='utf-8-sig')
print(f"저장 완료! 총 {len(df)}행")

저장 완료! 총 1868행
